# Compile Results (Clean)

Este notebook arma tablas de **worst-group accuracy** por *dataset × correlación × método* de forma limpia y extensible.
- Selecciona automáticamente la mejor **época por validación** (max `worst_group_acc` en `val.csv`) y usa la fila correspondiente de `test.csv`.
- Promedia sobre *seeds* (y opcionalmente añade `± std (n)`).
- Exporta CSV y Markdown de la tabla final.

In [6]:
from pathlib import Path
import wg_results

ROOTS = [Path(".")]
ONLY_METHODS = ["erm","rw_erm", "gdro", "erm_dfr","erm_gdro","erm_rw","erm_dfr_gdro","erm_dfr_rw"]  # exactos, sin colapsar

DATASET_CORRS = {
    "MNISTCIFAR": [0.0, 0.25, 0.5, 0.75, 0.9],
    "CUB":        [0.0, 0.25, 0.5, 0.75, 0.9],
    "CelebA":     [0.9],
    "MultiNLI":   [0.9],
    "civilcomments": [0.9],
}

raw = wg_results.crawl_experiments(
    ROOTS, split="test", selection="val_worst",
    allowed_methods=ONLY_METHODS,
    dataset_corrs=DATASET_CORRS,
    # corr_token_maps opcional; CUB ya tiene default:
    # corr_token_maps={"CUB":{"50":0.0,"625":0.25,"75":0.5,"875":0.75,"95":0.9,"100":1.0}},
    dataset_allowlist=[
    "MNISTCIFAR","CUB","CelebA","MultiNLI","civilcomments"]  # opcional
)
agg = wg_results.aggregate_by_seed(raw)
table = wg_results.pivot_table(agg, methods_order=ONLY_METHODS, value="worst_acc", include_std=True, as_percent=True)
print(table)



CUB/erm_50/model_outputs_111/test.csv
CUB/erm_50/model_outputs_333/test.csv
CUB/erm_50/model_outputs_222/test.csv
CUB/erm_625/model_outputs_111/test.csv
CUB/erm_625/model_outputs_333/test.csv
CUB/erm_625/model_outputs_222/test.csv
CUB/erm_75/model_outputs_111/test.csv
CUB/erm_75/model_outputs_333/test.csv
CUB/erm_75/model_outputs_222/test.csv
CUB/erm_875/model_outputs_111/test.csv
CUB/erm_875/model_outputs_333/test.csv
CUB/erm_875/model_outputs_222/test.csv
CUB/erm_95/model_outputs_111/test.csv
CUB/erm_95/model_outputs_333/test.csv
CUB/erm_95/model_outputs_222/test.csv
CUB/erm_dfr_625/model_outputs_111/test.csv
CUB/erm_dfr_625/model_outputs_333/test.csv
CUB/erm_dfr_625/model_outputs_222/test.csv
CUB/erm_dfr_75/model_outputs_111/test.csv
CUB/erm_dfr_75/model_outputs_333/test.csv
CUB/erm_dfr_75/model_outputs_222/test.csv
CUB/erm_dfr_875/model_outputs_111/test.csv
CUB/erm_dfr_875/model_outputs_333/test.csv
CUB/erm_dfr_875/model_outputs_222/test.csv
CUB/erm_dfr_95/model_outputs_111/test.cs

In [7]:
import re
import pandas as pd
from typing import List, Optional, Dict

def pivot_to_latex(
    piv: pd.DataFrame,
    *,
    # Tabla / entorno
    caption: Optional[str] = None,
    label: Optional[str] = None,
    table_env: str = "table*",
    tabcolsep_pt: int = 1,
    arraystretch: str = "1.0",
    booktabs: bool = True,
    # Columnas clave del pivote
    dataset_col: str = "dataset",
    corr_col: str = "correlacion",
    # Selección y orden de columnas (métodos)
    columns: Optional[List[str]] = None,            # ← elige qué métodos/columnas mostrar
    column_labels: Optional[Dict[str, str]] = None, # ← etiquetas a mostrar por columna (opcional)
    # Orden de filas
    dataset_order: Optional[List[str]] = None,      # ← orden explícito de datasets
    corr_order: Optional[List[float]] = None,       # ← orden explícito de correlaciones (e.g., [0.0,0.25,0.5,0.75,0.9])
    # Formato de celdas
    percent: bool = True,
    small_std_macro: Optional[str] = r"\st",        # define p.ej.: \newcommand{\st}[1]{\textcolor{black}{\scriptsize(#1)}}
    show_n: bool = False,                           # muestra (n) en pequeño
    use_multirow: bool = True,                      # dataset una sola vez por bloque
) -> str:
    """
    Renderiza a LaTeX el DataFrame 'piv' (salida de tu pivot_table):
      filas = (dataset, correlación)  |  columnas = métodos
    - Multirow por dataset (opcional).
    - booktabs y separación con \\midrule entre datasets.
    - Celdas aceptan "mean ± std (n)" o valores numéricos; std en pequeño con 'small_std_macro'.
    - 'columns' permite seleccionar y ordenar métodos/columnas a mostrar; 'column_labels' renombra en header.
    - 'dataset_order' y 'corr_order' controlan el orden de bloques/filas.
    """
    assert dataset_col in piv.columns and corr_col in piv.columns, \
        "piv debe tener columnas 'dataset' y 'correlacion'"

    # --- columnas de método a mostrar ---
    all_method_cols = [c for c in piv.columns if c not in (dataset_col, corr_col)]
    if columns is None:
        method_cols = all_method_cols
    else:
        method_cols = [c for c in columns if c in all_method_cols]  # respeta orden dado

    # --- orden de datasets ---
    df = piv.copy()
    if dataset_order is None:
        dataset_order = list(dict.fromkeys(df[dataset_col].tolist()))
    # Categórico para mantener el orden deseado
    df[dataset_col] = pd.Categorical(df[dataset_col], categories=dataset_order, ordered=True)

    # --- orden de correlaciones ---
    if corr_order is not None:
        df[corr_col] = pd.Categorical(df[corr_col], categories=corr_order, ordered=True)
    else:
        # intenta ordenar numéricamente si no se especifica
        try:
            df[corr_col] = df[corr_col].astype(float)
        except Exception:
            pass

    df = df.sort_values([dataset_col, corr_col], kind="mergesort")

    # --- formateo de celdas ---
    pm_pat  = re.compile(r"^\s*(-?\d+(?:\.\d+)?)\s*±\s*(-?\d+(?:\.\d+)?)\s*(?:\((\d+)\))?\s*$")
    num_pat = re.compile(r"^\s*-?\d+(?:\.\d+)?\s*$")

    def fmt_cell(x):
        if pd.isna(x):
            return "-"
        s = str(x).strip()
        # ya trae % -> intenta envolver std si es '±'
        if "%" in s:
            m = pm_pat.match(s.replace("%","").strip())
            if m:
                mean, std, n = m.groups()
                tail = (f"{small_std_macro}({float(std):.2f})" if small_std_macro
                        else "{\\small (" + f"{float(std):.2f}" + ")}")
                n_part = f" {small_std_macro}({n})" if (show_n and n is not None) else ""
                return f"{float(mean):.2f}\\% {tail}{n_part}"
            return s
        # "mean ± std (n)"
        m = pm_pat.match(s)
        if m:
            mean, std, n = m.groups()
            mean_part = f"{float(mean):.2f}\\%" if percent else f"{float(mean):.4f}"
            tail = (f"{small_std_macro}({float(std):.2f})" if small_std_macro
                    else "{\\small (" + f"{float(std):.2f}" + ")}")
            n_part = f" {small_std_macro}({n})" if (show_n and n is not None) else ""
            return f"{mean_part} {tail}{n_part}"
        # número simple
        if num_pat.match(s):
            val = float(s)
            return f"{val:.2f}\\%" if percent else f"{val:.4f}"
        return s

    # --- helpers ---
    def esc(name: str) -> str:
        return str(name).replace('_', r'\_')

    # --- encabezados ---
    col_align = "l l " + " ".join(["c"] * len(method_cols))
    lines = []
    lines.append(f"\\begin{{{table_env}}}[t]")
    if caption: lines.append(f"    \\caption{{{caption}}}")
    if label:   lines.append(f"    \\label{{{label}}}")
    lines.append("    \\centering")
    lines.append("\\begin{small}")
    lines.append(f"\\setlength{{\\tabcolsep}}{{{tabcolsep_pt}pt}}")
    lines.append(f"\\renewcommand{{\\arraystretch}}{{{arraystretch}}}")
    lines.append(f"\\begin{{tabular}}{{{col_align}}}")
    if booktabs: lines.append("\\toprule")

    header = [dataset_col.capitalize(), "Correlación"]
    if column_labels:
        header += [column_labels.get(c, c) for c in method_cols]
    else:
        header += method_cols
    hdr_line = " & ".join([("\\textbf{"+esc(h)+"}") for h in header]) + r" \\"
    lines.append(hdr_line)
    if booktabs: lines.append("\\midrule")

    # --- cuerpo con multirow correcto (sin & extra) ---
    groups = list(df.groupby(dataset_col, sort=False))
    for gi, (ds, block) in enumerate(groups):
        block = block.sort_values(corr_col, kind="mergesort")
        n_rows = len(block)
        ds_tex = esc(ds)
        for ridx, (_, row) in enumerate(block.iterrows()):
            # dataset cell
            if use_multirow:
                ds_cell = f"\\multirow{{{n_rows}}}{{*}}{{{ds_tex}}}" if ridx == 0 else ""
            else:
                ds_cell = ds_tex
            # corr cell (muestra como venga; si quieres un formateo especial, ajusta aquí)
            corr_str = str(row[corr_col])
            # method cells
            meth_cells = [fmt_cell(row.get(c, float('nan'))) for c in method_cols]
            # ensamblar la fila (nota: si ds_cell == "" la fila empieza directamente con '&', lo cual es correcto)
            cells = [ds_cell, corr_str] + meth_cells
            line = " & ".join(cells) + r" \\"
            lines.append(line)
        # separador entre datasets (no después del último)
        if booktabs and gi < len(groups) - 1:
            lines.append("\\midrule")

    if booktabs: lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{small}")
    lines.append(f"\\end{{{table_env}}}")
    return "\n".join(lines)



In [8]:
cols = ["erm", "rw_erm", "gdro", "erm_rw", "erm_gdro", "erm_dfr", "erm_dfr_rw", "erm_dfr_gdro"]  # solo estas columnas del pivote
labels = {"erm": "ERM","rw_erm": "RW", "gdro": "GDRO","erm_dfr": "SUBG", "erm_dfr_rw": "RW+", "erm_dfr_gdro": "GDRO+", "erm_rw":"RW-FT", "erm_gdro": "GDRO-FT"}  # opcional

tex = pivot_to_latex(
    table,
    caption="Worst Group Accuracy por dataset/correlación",
    label="tab:wg_summary",
    columns=cols,
    column_labels=labels,
    dataset_order=["MNISTCIFAR","CUB","CelebA","MultiNLI","civilcomments"],  # orden explícito
    corr_order=[0.0,0.25,0.5,0.75,0.9],           # orden explícito
    small_std_macro=r"\st",
    use_multirow=True
)
print(tex)


\begin{table*}[t]
    \caption{Worst Group Accuracy por dataset/correlación}
    \label{tab:wg_summary}
    \centering
\begin{small}
\setlength{\tabcolsep}{1pt}
\renewcommand{\arraystretch}{1.0}
\begin{tabular}{l l c c c c c c c c}
\toprule
\textbf{Dataset} & \textbf{Correlación} & \textbf{ERM} & \textbf{RW} & \textbf{GDRO} & \textbf{RW-FT} & \textbf{GDRO-FT} & \textbf{SUBG} & \textbf{RW+} & \textbf{GDRO+} \\
\midrule
\multirow{5}{*}{MNISTCIFAR} & 0.0 & 88.22\% \st(1.29) & 88.35\% \st(1.45) & 88.09\% \st(1.23) & 88.89\% \st(1.01) & 88.62\% \st(0.93) & - & - & - \\
 & 0.25 & 82.11\% \st(0.41) & 86.72\% \st(2.24) & 86.59\% \st(2.03) & 86.86\% \st(1.31) & 86.72\% \st(1.17) & 87.95\% \st(0.70) & 88.22\% \st(0.46) & 88.22\% \st(0.61) \\
 & 0.5 & 78.08\% \st(0.67) & 88.50\% \st(0.69) & 88.34\% \st(0.58) & 88.85\% \st(1.02) & 88.72\% \st(0.80) & 87.15\% \st(1.20) & 87.42\% \st(0.93) & 87.01\% \st(2.06) \\
 & 0.75 & 54.86\% \st(5.04) & 84.26\% \st(1.13) & 83.52\% \st(0.93) & 84.38\% \st(0.21) 

/tmp/ipykernel_2560593/1970280198.py:128: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = list(df.groupby(dataset_col, sort=False))


In [3]:
from pathlib import Path
from wg_results import crawl_experiments, aggregate_by_seed, pivot_table

ROOTS = [Path("CUB")]
SPLIT = "test"

# Solo quiero considerar estos métodos (tras alias):
ONLY_METHODS = ["erm", "rw", "groupdro", "subg", "gdro+","rw+"]  # <- allowlist

# Alias para normalizar nombres de carpeta:
ALIASES_EXACT = {
    "rw": "rw_erm",
    "groupdro": "gdro",
    "erm": "erm",
    "subg": "erm_dfr",
    "gdro+": "erm_dfr_gdro",
    "rw+": "erm_dfr_rw"
}
ALIASES_REGEX = [
    (r"^jtt_.*$", "jtt"),
    (r"^subg_.*$", "subg"),
]

# Opcional: lista “conocida” de métodos (limita la detección a estos).
# Si la dejas None, no te restringes a un set fijo.
KNOWN = ["erm", "rw", "groupdro", "subg", "gdro+","rw+"]   # o None

raw = crawl_experiments(
    ROOTS,
    split=SPLIT,
    selection="val_worst",
    method_aliases_exact=ALIASES_EXACT,
    method_aliases_regex=ALIASES_REGEX,
    allowed_methods=ONLY_METHODS,      # << solo estos métodos pasan
    known_methods=KNOWN                # << opcional; pon None para no restringir
)

agg = aggregate_by_seed(raw)
table = pivot_table(agg, methods_order=ONLY_METHODS, value="worst_acc", include_std=False, as_percent=True)
table


CUB


method,dataset,correlacion,erm,rw
0,CUB,0.0,87.816336,NaN
1,CUB,0.1,64.429654,NaN
2,CUB,0.25,71.786357,NaN
3,CUB,0.5,76.289394,NaN
4,CUB,0.75,79.361373,NaN
5,CUB,0.9,78.748969,NaN
6,CUB,1.0,84.272903,88.477395


In [2]:
!pwd

/workspace1/araymond/svdrop/results


In [2]:
# 1) Crawler: recorrer carpetas y recoger resultados por experimento
raw = crawl_experiments(ROOTS, split=SPLIT, selection=SELECTION, allowed_methods=ONLY_METHODS)
raw.sort_values(["dataset","correlacion","method","seed","epoch"], inplace=True, na_position="last")
raw.reset_index(drop=True, inplace=True)
raw.head(10)

KeyError: 'dataset'

In [ ]:
# 2) Agregar sobre seeds: promedio y std
agg = aggregate_by_seed(raw)
agg.sort_values(["dataset","correlacion","method"], inplace=True, na_position="last")
agg.reset_index(drop=True, inplace=True)
agg.head(10)

In [ ]:
# 3) Tabla final: (dataset, correlación) × métodos, usando worst_acc
table = pivot_table(agg, methods_order=METHODS_ORDER, value="worst_acc",
                    include_std=INCLUDE_STD, as_percent=AS_PERCENT)
table

In [ ]:
# 4) Guardar CSV y Markdown
paths = save_outputs(table, OUT_PREFIX)
paths

## Ablation over Dataset size for RW+,GDRO+

In [17]:
from pathlib import Path
import wg_results

ROOTS = [Path(".")]
ONLY_METHODS = ["erm_rw_0.1", "erm_rw_0.25", "erm_rw_0.5", "erm_rw_0.75", "erm_rw_0.9",
                "erm_gdro_0.1", "erm_gdro_0.25", "erm_gdro_0.5", "erm_gdro_0.75", "erm_gdro_0.9",
                "erm_dfr_rw_0.1", "erm_dfr_rw_0.25", "erm_dfr_rw_0.5", "erm_dfr_rw_0.75", "erm_dfr_rw_0.9",
                "erm_dfr_gdro_0.1", "erm_dfr_gdro_0.25", "erm_dfr_gdro_0.5", "erm_dfr_gdro_0.75", "erm_dfr_gdro_0.9",
                "erm_dfr_rw_new_0.1", "erm_dfr_rw_new_0.25", "erm_dfr_rw_new_0.5", "erm_dfr_rw_new_0.75", "erm_dfr_rw_new_0.9",
                "erm_dfr_gdro_new_0.1", "erm_dfr_gdro_new_0.25", "erm_dfr_gdro_new_0.5", "erm_dfr_gdro_new_0.75", "erm_dfr_gdro_new_0.9"]  # exactos, sin colapsar

DATASET_CORRS = {
    "MNISTCIFAR": [0.0, 0.25, 0.5, 0.75, 0.9],
    "CUB":        [0.0, 0.25, 0.5, 0.75, 0.9],
    "CelebA":     [0.9],

}

raw = wg_results.crawl_experiments(
    ROOTS, split="test", selection="val_worst",
    allowed_methods=ONLY_METHODS,
    dataset_corrs=DATASET_CORRS,
    # corr_token_maps opcional; CUB ya tiene default:
    corr_token_maps={"CUB":{"50":0.0,"625":0.25,"75":0.5,"875":0.75,"95":0.9,"100":1.0}},
    dataset_allowlist=[
    "MNISTCIFAR","CUB","CelebA"]  # opcional
)
agg = wg_results.aggregate_by_seed(raw)
table = wg_results.pivot_table(agg, methods_order=ONLY_METHODS, value="worst_acc", include_std=True, as_percent=True)
print(table)



CUB/erm_dfr_gdro_new_0.1_95/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_0.1_95/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_0.1_95/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_0.25_95/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_0.25_95/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_0.25_95/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_0.5_95/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_0.5_95/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_0.5_95/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_0.75_95/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_0.75_95/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_0.75_95/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_0.9_95/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_0.9_95/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_0.9_95/model_outputs_222/test.csv
CUB/erm_dfr_rw_new_0.1_95/model_outputs_111/test.csv
CUB/erm_dfr_rw_new_0.1_95/model_outputs_333/test.csv
CUB/erm_dfr_rw_new_0.1_95/model_outputs_222/test.csv
CUB/erm_df

In [18]:
agg

,dataset,correlacion,method,worst_acc,avg_acc,worst_acc_std,avg_acc_std,n
0,CUB,0.9,erm_dfr_gdro_new_0.1,0.798522,0.869578,0.074184,0.046538,3
1,CUB,0.9,erm_dfr_gdro_new_0.25,0.843257,0.894546,0.026574,0.015270,3
2,CUB,0.9,erm_dfr_gdro_new_0.5,0.867547,0.904211,0.008187,0.010796,3
3,CUB,0.9,erm_dfr_gdro_new_0.75,0.882139,0.909907,0.007840,0.011899,3
4,CUB,0.9,erm_dfr_gdro_new_0.9,0.889173,0.905247,0.003474,0.008870,3
5,CUB,0.9,erm_dfr_rw_new_0.1,0.792166,0.866298,0.071913,0.046627,3
6,CUB,0.9,erm_dfr_rw_new_0.25,0.838440,0.896790,0.021586,0.016744,3
7,CUB,0.9,erm_dfr_rw_new_0.5,0.866433,0.903521,0.006505,0.007926,3
8,CUB,0.9,erm_dfr_rw_new_0.75,0.880062,0.908469,0.008242,0.009381,3
9,CUB,0.9,erm_dfr_rw_new_0.9,0.889927,0.908699,0.005470,0.004497,3


In [19]:
import pandas as pd
import numpy as np

def make_alpha_grid_latex(
    df: pd.DataFrame,
    metric: str = "worst_acc",          # o "avg_acc"
    show_std: bool = True,              # incluye (std) si True
    percent: bool = True,               # multiplica x100 y agrega %
    bold_best: bool = True,             # pone en negrita el mejor por columna dentro de cada dataset
    alpha_columns = (0.1,0.25,0.5,0.75,0.9,1.0),  # orden deseado de columnas
    dataset_order = ("MNISTCIFAR","CUB","CelebA"),# ajusta a lo que tengas
    family_order = ("erm","erm_rw","erm_gdro","erm_dfr_rw","ern_dfr_rw_new","erm_dfr_gdro_new"),
    dataset_names = {"MNISTCIFAR":"MNIST--CIFAR","CUB":"Waterbirds","CelebA":"CelebA"},
    family_names = {
        "erm":"ERM",
        "erm_rw":"RW-FT",
        "erm_gdro":"GDRO-FT",
        "erm_dfr_rw":"RW+",
        "erm_dfr_gdro":"GDRO+",
        "erm_dfr_rw_new":"RW+",
        "erm_dfr_gdro_new":"GDRO+",
    },
    caption: str = "Worst-group accuracy (\\%) across alphas.",
    label: str = "tab:alphagrid",
    decimals_val: int = 2,
    decimals_std: int = 2,
):
    """
    Convierte df en una tabla LaTeX estilo 'foto':
    Dataset | Method | 0.1 | 0.25 | 0.5 | 0.75 | 0.9 | 1.0
    """
    df = df.copy()

    # --- parsea family y alpha desde el campo 'method' ---
    def _parse_method(m):
        parts = str(m).split("_")
        try:
            alpha = float(parts[-1])
            family = "_".join(parts[:-1])
        except ValueError:
            alpha = np.nan
            family = str(m)
        return pd.Series({"family": family, "alpha": alpha})

    parsed = df["method"].apply(_parse_method)
    df = pd.concat([df, parsed], axis=1)

    # por si hay datasets fuera del orden propuesto:
    dataset_order = [d for d in dataset_order if d in df["dataset"].unique()] + \
                    [d for d in df["dataset"].unique() if d not in dataset_order]

    # columnas del std
    std_col = f"{metric}_std"

    # helper de formato
    def fmt(mean, std):
        if pd.isna(mean):
            return "--"
        v = mean * 100 if percent else mean
        s = std * 100 if (show_std and not pd.isna(std)) else None
        if s is None:
            return f"{v:.{decimals_val}f}\\%"
        else:
            return f"{v:.{decimals_val}f}\\% {{\\scriptsize ({s:.{decimals_std}f})}}"

    # precomputar diccionario {(dataset,family,alpha) -> (mean,std)}
    key2val = {}
    for _, r in df.iterrows():
        key = (r["dataset"], r["family"], float(r["alpha"]) if not pd.isna(r["alpha"]) else None)
        key2val[key] = (float(r[metric]), float(r[std_col]) if std_col in df.columns else np.nan)

    # por dataset, calculamos mejor por columna (para bold)
    best_mask = {}
    if bold_best:
        for ds in dataset_order:
            # familias presentes en ds
            fams = sorted(df.loc[df["dataset"]==ds, "family"].unique().tolist(),
                          key=lambda f: (family_order.index(f) if f in family_order else 999, f))
            for a in alpha_columns:
                best_val = -np.inf
                for f in fams:
                    val = key2val.get((ds,f,a), (np.nan,np.nan))[0]
                    if not pd.isna(val) and val > best_val:
                        best_val = val
                if best_val > -np.inf:
                    best_mask[(ds,a)] = best_val

    # construcción LaTeX
    colspec = "l" + "l" + "c"*len(alpha_columns)  # Dataset, Method, alphas...
    lines = []
    lines += [
        "\\begin{table}[t]",
        "\\centering",
        "\\small",
        f"\\begin{{tabular}}{{{colspec}}}",
        "\\toprule",
        "Dataset & Method & " + " & ".join([f"{a:g}" for a in alpha_columns]) + " \\\\",
        "\\midrule",
    ]

    for ds in dataset_order:
        sub = df[df["dataset"]==ds]
        if sub.empty:
            continue
        # familias presentes (en orden deseado)
        fams_present = [f for f in family_order if f in sub["family"].unique()]
        if not fams_present:
            continue

        ds_nice = dataset_names.get(ds, ds)
        nrows = len(fams_present)

        for i, fam in enumerate(fams_present):
            row = []
            row.append(f"\\multirow{{{nrows}}}{{*}}{{{ds_nice}}}" if i==0 else "")
            row.append(family_names.get(fam, fam))

            for a in alpha_columns:
                mean, std = key2val.get((ds, fam, a), (np.nan, np.nan))
                cell = fmt(mean, std)

                # bold del mejor por columna en este dataset
                if bold_best and (ds, a) in best_mask and not pd.isna(mean):
                    if abs(mean - best_mask[(ds,a)]) < 1e-12:
                        cell = f"\\textbf{{{cell}}}"
                row.append(cell)

            lines.append(" & ".join(row) + " \\\\")
        lines.append("\\midrule")

    lines += [
        "\\bottomrule",
        "\\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\end{table}",
    ]
    return "\n".join(lines)
latex = make_alpha_grid_latex(
    agg,
    metric="worst_acc",            # o "avg_acc"
    show_std=True,                 # std opcional
    percent=True,
    bold_best=True,
    alpha_columns=(0.1,0.25,0.5,0.75,0.9),
    dataset_order=("MNISTCIFAR","CUB","CelebA"),    # agrega otros si tienes
    caption="Worst-group accuracy (\\%, mean {\\scriptsize (std)}) across alphas.",
    label="tab:alpha_worst"
)
print(latex)

\begin{table}[t]
\centering
\small
\begin{tabular}{llccccc}
\toprule
Dataset & Method & 0.1 & 0.25 & 0.5 & 0.75 & 0.9 \\
\midrule
\multirow{3}{*}{MNIST--CIFAR} & RW-FT & \textbf{80.63\% {\scriptsize (1.19)}} & 79.33\% {\scriptsize (1.20)} & 78.88\% {\scriptsize (2.79)} & 79.28\% {\scriptsize (2.07)} & \textbf{80.40\% {\scriptsize (0.81)}} \\
 & GDRO-FT & 78.77\% {\scriptsize (2.54)} & 78.53\% {\scriptsize (1.46)} & 78.01\% {\scriptsize (2.11)} & 78.96\% {\scriptsize (0.75)} & 78.22\% {\scriptsize (1.51)} \\
 & RW+ & 75.85\% {\scriptsize (5.13)} & 79.79\% {\scriptsize (1.81)} & \textbf{80.59\% {\scriptsize (1.01)}} & \textbf{80.72\% {\scriptsize (0.40)}} & 79.12\% {\scriptsize (3.14)} \\
\midrule
\multirow{3}{*}{Waterbirds} & RW-FT & 49.43\% {\scriptsize (1.04)} & 56.28\% {\scriptsize (2.16)} & 62.88\% {\scriptsize (3.74)} & 66.20\% {\scriptsize (2.36)} & 66.25\% {\scriptsize (0.80)} \\
 & GDRO-FT & 53.32\% {\scriptsize (1.72)} & 62.36\% {\scriptsize (3.28)} & 67.68\% {\scriptsize (0.70

In [20]:
agg

,dataset,correlacion,method,worst_acc,avg_acc,worst_acc_std,avg_acc_std,n
0,CUB,0.9,erm_dfr_gdro_new_0.1,0.798522,0.869578,0.074184,0.046538,3
1,CUB,0.9,erm_dfr_gdro_new_0.25,0.843257,0.894546,0.026574,0.015270,3
2,CUB,0.9,erm_dfr_gdro_new_0.5,0.867547,0.904211,0.008187,0.010796,3
3,CUB,0.9,erm_dfr_gdro_new_0.75,0.882139,0.909907,0.007840,0.011899,3
4,CUB,0.9,erm_dfr_gdro_new_0.9,0.889173,0.905247,0.003474,0.008870,3
5,CUB,0.9,erm_dfr_rw_new_0.1,0.792166,0.866298,0.071913,0.046627,3
6,CUB,0.9,erm_dfr_rw_new_0.25,0.838440,0.896790,0.021586,0.016744,3
7,CUB,0.9,erm_dfr_rw_new_0.5,0.866433,0.903521,0.006505,0.007926,3
8,CUB,0.9,erm_dfr_rw_new_0.75,0.880062,0.908469,0.008242,0.009381,3
9,CUB,0.9,erm_dfr_rw_new_0.9,0.889927,0.908699,0.005470,0.004497,3


## Unfreezing layers

In [23]:
from pathlib import Path
import wg_results

ROOTS = [Path(".")]
ONLY_METHODS = ["erm", "rw_erm", "gdro", "erm_rw", "erm_gdro", "erm_dfr_rw_new", "erm_dfr_gdro_new",
                "erm_gdro_frz0", "erm_gdro_frz1", "erm_gdro_frz2", "erm_gdro_frz3", "erm_gdro_frz4",
                "erm_rw_frz0", "erm_rw_frz1", "erm_rw_frz2", "erm_rw_frz3", "erm_rw_frz4",
                "erm_gdro_frz_rst1", "erm_gdro_frz_rst2", "erm_gdro_frz_rst3", "erm_gdro_frz_rst4",
                "erm_rw_frz_rst1", "erm_rw_frz_rst2", "erm_rw_frz_rst3", "erm_rw_frz_rst4",
               ]  # exactos, sin colapsar

DATASET_CORRS = {
    "CUB":        [0.0, 0.25, 0.5, 0.75, 0.9]
}

raw = wg_results.crawl_experiments(
    ROOTS, split="test", selection="val_worst",
    allowed_methods=ONLY_METHODS,
    dataset_corrs=DATASET_CORRS,
    # corr_token_maps opcional; CUB ya tiene default:
    corr_token_maps={"CUB":{"50":0.0,"625":0.25,"75":0.5,"875":0.75,"95":0.9,"100":1.0}},
    dataset_allowlist=[
  "CUB"]  # opcional
)
agg = wg_results.aggregate_by_seed(raw)
table = wg_results.pivot_table(agg, methods_order=ONLY_METHODS, value="worst_acc", include_std=True, as_percent=True)
print(table)


CUB/erm_50/model_outputs_111/test.csv
CUB/erm_50/model_outputs_333/test.csv
CUB/erm_50/model_outputs_222/test.csv
CUB/erm_625/model_outputs_111/test.csv
CUB/erm_625/model_outputs_333/test.csv
CUB/erm_625/model_outputs_222/test.csv
CUB/erm_75/model_outputs_111/test.csv
CUB/erm_75/model_outputs_333/test.csv
CUB/erm_75/model_outputs_222/test.csv
CUB/erm_875/model_outputs_111/test.csv
CUB/erm_875/model_outputs_333/test.csv
CUB/erm_875/model_outputs_222/test.csv
CUB/erm_95/model_outputs_111/test.csv
CUB/erm_95/model_outputs_333/test.csv
CUB/erm_95/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_625/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_625/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_625/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_75/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_75/model_outputs_333/test.csv
CUB/erm_dfr_gdro_new_75/model_outputs_222/test.csv
CUB/erm_dfr_gdro_new_875/model_outputs_111/test.csv
CUB/erm_dfr_gdro_new_875/model_outputs_333/test.csv
CUB/erm_dfr

In [30]:
import pandas as pd
import numpy as np

def latex_corr_grid_by_dataset(
    df: pd.DataFrame,
    dataset: str = "CUB",
    metric: str = "worst_acc",      # o "avg_acc"
    show_std: bool = True,
    percent: bool = True,
    decimals_val: int = 2,
    decimals_std: int = 2,
    corr_order: tuple | None = None,
    # ----- BLOQUES Y NOMBRES BONITOS (ajústalos a gusto) -----
    blocks_spec: list[tuple] | None = None,  # lista de ( [metodos_en_orden], [etiquetas_lindas] )
    caption: str | None = None,
    label: str = "tab:corrgrid",
):
    """
    Genera una tabla LaTeX estilo 'foto 2':
    Cols = correlaciones; Filas = métodos (agrupados en bloques); Tabla por dataset.

    blocks_spec: lista de bloques; cada bloque es (methods, labels)
        - methods: lista de IDs tal como aparecen en df['method']
        - labels:  lista de nombres lindos para imprimir en la primera columna
      Los métodos que no existan en el DF se omiten automáticamente.
    """
    # --- filtrar dataset ---
    sub = df[df["dataset"] == dataset].copy()
    if sub.empty:
        raise ValueError(f"No hay filas para dataset='{dataset}'")

    # --- columnas disponibles (orden) ---
    if corr_order is None:
        corr_order = tuple(sorted(sub["correlacion"].unique()))
    corr_order = tuple(corr_order)

    # --- helper de formato de celdas ---
    std_col = f"{metric}_std"

    def fmt_cell(mean, std):
        if pd.isna(mean):
            return "-"
        v = mean * 100 if percent else mean
        if show_std and not pd.isna(std):
            s = std * 100 if percent else std
            return f"{v:.{decimals_val}f}\\% {{\\scriptsize ({s:.{decimals_std}f})}}"
        else:
            return f"{v:.{decimals_val}f}\\%"

    # --- pivot de mean y std ---
    piv_mean = sub.pivot_table(index="method", columns="correlacion", values=metric, aggfunc="mean")
    piv_std  = sub.pivot_table(index="method", columns="correlacion", values=std_col, aggfunc="mean")

    # --- bloques por defecto, basados en tus nombres de método ---
    present = set(piv_mean.index.tolist())

    def pick(methods, labels):
        ms, ls = [], []
        for m, l in zip(methods, labels):
            if m in present:
                ms.append(m); ls.append(l)
        return ms, ls

    if blocks_spec is None:
        # BLOQUE 1: Baselines (como en la parte superior de tu imagen)
        b1_m = ["erm", "rw_erm", "gdro"]
        b1_l = ["ERM", "RW", "GDRO"]

        # BLOQUE 2: GDRO-FT (FREE/FRZx/RSTx)
        frz_gdro = [f"erm_gdro_frz{i}" for i in range(5)]
        rst_gdro = [f"erm_gdro_frz_rst{i}" for i in range(1,5)]
        b2_m = ["erm_gdro"] + frz_gdro + rst_gdro
        b2_l = (["GDRO-FT (FREE)"] +
                [f"GDRO-FT (FRZ{i})" for i in range(5)] +
                [f"GDRO-FT (RST{i})" for i in range(1,5)])

        # BLOQUE 3: ERM-FT (FREE/FRZx/RSTx)  (en tu DF estos vienen como 'erm_rw_*')
        frz_rw = [f"erm_rw_frz{i}" for i in range(5)]
        rst_rw = [f"erm_rw_frz_rst{i}" for i in range(1,5)]
        b3_m = ["erm_rw"] + frz_rw + rst_rw
        b3_l = (["ERM-FT (FREE)"] +
                [f"ERM-FT (FRZ{i})" for i in range(5)] +
                [f"ERM-FT (RST{i})" for i in range(1,5)])

        # BLOQUE 4: DFR+ variantes
        b4_m = ["erm_dfr_rw_new", "erm_dfr_gdro_new"]
        b4_l = ["RW+", "GDRO+"]

        blocks_spec = [pick(b1_m, b1_l), pick(b2_m, b2_l), pick(b3_m, b3_l), pick(b4_m, b4_l)]

    # --- función para bold del mejor por columna dentro de un bloque ---
    def bold_block_cells(rows_idx):
        # retorna set de (method, corr) que deben ir en bold
        winners = set()
        for c in corr_order:
            best_val = -np.inf
            # buscar mejor mean en la columna c, dentro del bloque
            for m in rows_idx:
                val = piv_mean.loc[m, c] if (m in piv_mean.index and c in piv_mean.columns) else np.nan
                if not pd.isna(val) and val > best_val:
                    best_val = val
            if best_val == -np.inf:
                continue
            for m in rows_idx:
                val = piv_mean.loc[m, c] if (m in piv_mean.index and c in piv_mean.columns) else np.nan
                if not pd.isna(val) and abs(val - best_val) < 1e-12:
                    winners.add((m, c))
        return winners

    # --- construir LaTeX ---
    ncols = 1 + len(corr_order)  # 1 columna 'METHOD' + alphas
    colspec = "l" + "c"*len(corr_order)

    lines = []
    lines += [
        "\\begin{table}[t]",
        "\\centering",
        "\\small",
        f"\\begin{{tabular}}{{{colspec}}}",
        "\\toprule",
        "METHOD & " + " & ".join([f"{c:g}" for c in corr_order]) + " \\\\",
        "\\midrule",
    ]

    # recorrer bloques
    for methods, labels in blocks_spec:
        if not methods:
            continue

        # mejores por columna dentro del bloque
        bold_set = bold_block_cells(methods)

        for m, label in zip(methods, labels):
            row_cells = [label]
            for c in corr_order:
                mu = piv_mean.loc[m, c] if (m in piv_mean.index and c in piv_mean.columns) else np.nan
                sd = piv_std.loc[m, c] if (m in piv_std.index and c in piv_std.columns) else np.nan
                cell = fmt_cell(mu, sd)
                if (m, c) in bold_set and cell != "-":
                    cell = f"\\textbf{{{cell}}}"
                row_cells.append(cell)
            lines.append(" & ".join(row_cells) + " \\\\")
        lines.append("\\midrule")

    lines += [
        "\\bottomrule",
        "\\end{tabular}",
    ]
    cap = caption or (f"{'Worst-group' if metric=='worst_acc' else 'Average'} accuracy (\\%, "
                      + ("mean {\\scriptsize (std)})" if show_std else "mean)")
                      + f" on {dataset}.")
    lines += [f"\\caption{{{cap}}}", f"\\label{{{label}}}", "\\end{table}"]

    return "\n".join(lines)


In [31]:
latex = latex_corr_grid_by_dataset(
    agg,                      # tu DataFrame agregado por semilla
    dataset="CUB",
    metric="worst_acc",       # o "avg_acc"
    show_std=True,            # para ver (std) en scriptsize
    corr_order=(0.0,0.25,0.5,0.75,0.9),   # columnas como en la foto
    # Si quieres ajustar bloques o nombres, pasa blocks_spec personalizado (ver abajo)
    caption="Worst-group accuracy (\\%, mean {\\scriptsize (std)}) across correlations on Waterbirds.",
    label="tab:wb_alpha_grid"
)
print(latex)

\begin{table}[t]
\centering
\small
\begin{tabular}{lccccc}
\toprule
METHOD & 0 & 0.25 & 0.5 & 0.75 & 0.9 \\
\midrule
ERM & 89.20\% {\scriptsize (0.39)} & 89.56\% {\scriptsize (0.68)} & 87.44\% {\scriptsize (1.04)} & 80.74\% {\scriptsize (0.80)} & 71.96\% {\scriptsize (0.82)} \\
RW & 89.20\% {\scriptsize (0.50)} & \textbf{89.87\% {\scriptsize (1.53)}} & 88.84\% {\scriptsize (0.99)} & \textbf{88.06\% {\scriptsize (1.64)}} & \textbf{86.31\% {\scriptsize (0.30)}} \\
GDRO & \textbf{90.50\% {\scriptsize (0.41)}} & 89.50\% {\scriptsize (3.08)} & \textbf{90.13\% {\scriptsize (0.50)}} & 87.59\% {\scriptsize (0.39)} & 85.87\% {\scriptsize (0.86)} \\
\midrule
GDRO-FT (FREE) & 86.71\% {\scriptsize (0.50)} & 86.45\% {\scriptsize (0.78)} & 84.48\% {\scriptsize (0.36)} & 78.50\% {\scriptsize (1.22)} & 73.68\% {\scriptsize (3.06)} \\
GDRO-FT (FRZ0) & 88.79\% {\scriptsize (1.77)} & 91.17\% {\scriptsize (0.63)} & 89.77\% {\scriptsize (0.32)} & 87.49\% {\scriptsize (0.86)} & 82.96\% {\scriptsize (1.47)} 